# Export prompts for manual distractor curation

This notebook reads the first `n` text prompts from every benchmark dataset registered in `benchmark_choices/datasets.json`, then writes each correct prompt plus three blank distractor slots to `benchmark_choices/type_e/` or `benchmark_choices/type_g/`.

It runs from an existing checkout or clones the repository automatically on RunPod/Colab/Kaggle. Set `HF_TOKEN` for gated Hugging Face datasets. ShareGPT-4o-Image uses local metadata when `SHAREGPT4O_IMAGE_ROOT` is set; otherwise the notebook reports and skips it.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import tarfile
from pathlib import Path
from typing import Any


# Set this when the notebook is running outside the repository tree, for example:
# REPO_ROOT_OVERRIDE = Path("/content/transformers")  # Google Colab
# REPO_ROOT_OVERRIDE = Path("/kaggle/working/transformers")  # Kaggle
REPO_ROOT_OVERRIDE: Path | None = None

# Repository to clone when no local checkout can be found. Change this if the
# notebook belongs to a fork or private repository. Private repositories need
# suitable Git credentials in the notebook environment.
REPO_URL = "https://github.com/samhatescoding/transformers.git"
if Path("/workspace").is_dir():
    CLONE_DIR = Path("/workspace/transformers")  # RunPod
elif Path("/content").is_dir():
    CLONE_DIR = Path("/content/transformers")  # Colab
elif Path("/kaggle/working").is_dir():
    CLONE_DIR = Path("/kaggle/working/transformers")
else:
    CLONE_DIR = Path.cwd() / "transformers"


def is_repo_root(path: Path) -> bool:
    return (path / "benchmarks").is_dir() and (path / "dataset").is_dir()


def find_repo_root(start: Path | None = None, override: Path | None = None) -> Path | None:
    if override is not None:
        candidate = override.expanduser().resolve()
        if is_repo_root(candidate):
            return candidate
        raise FileNotFoundError(
            f"REPO_ROOT_OVERRIDE does not contain benchmarks/ and dataset/: {candidate}"
        )

    current = (start or Path.cwd()).resolve()
    candidates = [current, *current.parents]
    for base in (Path("/content"), Path("/kaggle/working"), Path.home()):
        if base.exists():
            candidates.extend(path for path in base.glob("*") if path.is_dir())
    for candidate in candidates:
        if is_repo_root(candidate):
            return candidate.resolve()
    return None


def clone_repo(repo_url: str, clone_dir: Path) -> Path:
    clone_dir = clone_dir.expanduser().resolve()
    if clone_dir.exists() and any(clone_dir.iterdir()):
        raise FileExistsError(
            f"Clone destination already exists and is not empty: {clone_dir}. "
            "Set CLONE_DIR to an empty/nonexistent path or REPO_ROOT_OVERRIDE to an existing checkout."
        )
    clone_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", repo_url, str(clone_dir)],
        check=True,
    )
    if not is_repo_root(clone_dir):
        raise FileNotFoundError(
            f"The cloned repository does not contain benchmarks/ and dataset/: {clone_dir}"
        )
    return clone_dir


REPO_ROOT = find_repo_root(override=REPO_ROOT_OVERRIDE)
if REPO_ROOT is None:
    print(f"No local checkout found. Cloning {REPO_URL} into {CLONE_DIR}...")
    REPO_ROOT = clone_repo(REPO_URL, CLONE_DIR)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")

In [ ]:
# Configuration
n = 20
CHOICE_ROOT = REPO_ROOT / "benchmark_choices"
TYPE_E_DATASETS = {"hq_edit", "imgedit", "magicbrush", "sharegpt4o_image_edit"}
TYPE_G_DATASETS = {"blip3o_60k", "conceptual_captions", "diffusiondb", "openvid1m", "sharegpt4o_image"}
DATASET_OUTPUT_DIRS = {
    **{name: CHOICE_ROOT / "type_e" for name in TYPE_E_DATASETS},
    **{name: CHOICE_ROOT / "type_g" for name in TYPE_G_DATASETS},
}
OVERWRITE = False  # Keep False after manual editing has started.

# The Hugging Face libraries automatically read HF_TOKEN from the environment.
# To set it only for this kernel, uncomment the next line.
# os.environ["HF_TOKEN"] = "hf_..."

if n < 1:
    raise ValueError("n must be at least 1.")

print("HF_TOKEN is set." if os.getenv("HF_TOKEN") else "HF_TOKEN is not set; public datasets may still work.")

In [ ]:
%pip install -q datasets huggingface_hub

from datasets import load_dataset
from huggingface_hub import hf_hub_download, hf_hub_url


# Each source deliberately targets text-only metadata. No image archives or
# image-valued Parquet columns are read.
DATASET_SPECS: dict[str, dict[str, Any]] = {
    "blip3o_60k": {
        "dataset_id": "BLIP3o/BLIP3o-60k",
        "source_type": "text_lines",
        "filename": "dalle3.txt",
    },
    "diffusiondb": {
        "dataset_id": "poloclub/diffusiondb",
        "source_type": "parquet_column",
        "filename": "metadata.parquet",
        "column": "prompt",
    },
    "hq_edit": {
        "dataset_id": "UCSC-VLAA/HQ-Edit",
        "source_type": "parquet_column",
        "filename": "data/HQ-Edit_0.parquet",
        "column": "edit",
    },
    "imgedit": {
        "dataset_id": "sysuyy/ImgEdit",
        "source_type": "tar_json",
        "filename": "Benchmark.tar",
        "member": "Benchmark/singleturn/singleturn.json",
        "column": "prompt",
    },
    "magicbrush": {
        "dataset_id": "osunlp/MagicBrush",
        "source_type": "parquet_column",
        "filename": "data/train-00000-of-00051-9fd9f23e2b1cb397.parquet",
        "column": "instruction",
    },
    "sharegpt4o_image": {
        "dataset_id": "hanlincs/sharegpt4oimage_processed",
        "source_type": "hf_dataset",
        "column": "txt",
    },
}

registry_path = REPO_ROOT / "benchmark_choices" / "datasets.json"
if registry_path.exists():
    configured_datasets = set(json.loads(registry_path.read_text(encoding="utf-8")))
    missing = configured_datasets - DATASET_SPECS.keys()
    if missing:
        raise RuntimeError(f"Add dataset adapters to this notebook for: {sorted(missing)}")
    DATASET_SPECS = {
        name: spec
        for name, spec in DATASET_SPECS.items()
        if name in configured_datasets
    }

print(f"Found {len(DATASET_SPECS)} datasets:")
for name, spec in sorted(DATASET_SPECS.items()):
    print(f"- {name}: {spec.get('dataset_id', 'local metadata')}")

In [ ]:
def hub_download(spec: dict[str, Any]) -> Path:
    return Path(
        hf_hub_download(
            spec["dataset_id"],
            spec["filename"],
            repo_type="dataset",
            token=os.getenv("HF_TOKEN"),
        )
    )


def load_prompts(spec: dict[str, Any], row_count: int) -> list[str]:
    source_type = spec["source_type"]
    if source_type == "text_lines":
        lines = hub_download(spec).read_text(encoding="utf-8").splitlines()
        return [line.strip() for line in lines if line.strip()][:row_count]

    if source_type == "json":
        payload = json.loads(hub_download(spec).read_text(encoding="utf-8"))
        records = payload if isinstance(payload, list) else payload.get("data", payload.get("samples", []))
        prompts = [str(row.get(spec["column"], "")).strip() for row in records if isinstance(row, dict)]
        return [prompt for prompt in prompts if prompt][:row_count]

    if source_type == "tar_json":
        with tarfile.open(hub_download(spec)) as archive:
            member = archive.extractfile(spec["member"])
            if member is None:
                raise FileNotFoundError(f"Missing {spec['member']} in {spec['filename']}")
            payload = json.loads(member.read().decode("utf-8"))
        records = payload.values() if isinstance(payload, dict) else payload
        prompts = [str(row.get(spec["column"], "")).strip() for row in records if isinstance(row, dict)]
        return [prompt for prompt in prompts if prompt][:row_count]

    if source_type == "parquet_column":
        url = hf_hub_url(spec["dataset_id"], spec["filename"], repo_type="dataset")
        dataset = load_dataset(
            "parquet",
            data_files={"train": url},
            split="train",
            streaming=True,
            columns=[spec["column"]],
        )
        prompts: list[str] = []
        for row in dataset:
            prompt = str(row.get(spec["column"], "")).strip()
            if prompt:
                prompts.append(prompt)
            if len(prompts) >= row_count:
                break
        return prompts

    if source_type == "hf_dataset":
        dataset = load_dataset(
            spec["dataset_id"],
            split="train",
            streaming=True,
            token=os.getenv("HF_TOKEN"),
        )
        prompts: list[str] = []
        for row in dataset:
            prompt = str(row.get(spec["column"], "")).strip()
            if prompt:
                prompts.append(prompt)
            if len(prompts) >= row_count:
                break
        return prompts

    raise ValueError(f"Unknown source_type: {source_type}")


def export_dataset(name: str, spec: dict[str, Any], row_count: int) -> dict[str, Any]:
    output_dir = DATASET_OUTPUT_DIRS[name]
    output_path = output_dir / f"{name}.json"
    if output_path.exists() and not OVERWRITE:
        return {
            "dataset": name,
            "status": "skipped_existing",
            "path": str(output_path),
        }

    prompts = load_prompts(spec, row_count)
    if len(prompts) < row_count:
        raise ValueError(f"Requested {row_count} prompts but found only {len(prompts)}.")

    exported_rows = []
    for row_index, correct_prompt in enumerate(prompts):
        exported_rows.append(
            {
                "row_index": row_index,
                "correct_prompt": correct_prompt,
                "distractors": ["", "", ""],
            }
        )

    payload = {
        "dataset": name,
        "source": spec.get("dataset_id", "local ShareGPT-4o-Image metadata"),
        "split": "train",
        "requested_rows": row_count,
        "exported_rows": len(exported_rows),
        "rows": exported_rows,
    }
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    return {
        "dataset": name,
        "status": "written",
        "rows": len(exported_rows),
        "path": str(output_path),
    }


for output_dir in set(DATASET_OUTPUT_DIRS.values()):
    output_dir.mkdir(parents=True, exist_ok=True)
results: list[dict[str, Any]] = []

for name, spec in sorted(DATASET_SPECS.items()):
    try:
        result = export_dataset(name, spec, n)
    except Exception as exc:
        result = {
            "dataset": name,
            "status": "error",
            "error": f"{type(exc).__name__}: {exc}",
        }
    results.append(result)
    print(json.dumps(result, ensure_ascii=False))

In [ ]:
written = [item for item in results if item["status"] == "written"]
skipped = [item for item in results if item["status"] == "skipped_existing"]
errors = [item for item in results if item["status"] == "error"]

print(f"Written: {len(written)} | Existing: {len(skipped)} | Errors: {len(errors)}")
if errors:
    print("\nDatasets needing attention:")
    for item in errors:
        print(f"- {item['dataset']}: {item['error']}")

print(f"\nEdit the generated JSON files in: {CHOICE_ROOT / 'type_e'} and {CHOICE_ROOT / 'type_g'}")